# Challenge 3: Human-in-the-Loop & Resilience

## Why HITL Matters

Your workflow routes correctly — but would you trust it to `restart_pod` in production
at 3 AM without asking anyone?

MAF provides:
- `ctx.request_info()` — pauses the workflow and asks the caller for input
- `@tool(approval_mode="always_require")` — tool calls trigger approval events
- `workflow.run(responses={...})` — resumes a paused workflow with answers
- `@response_handler` — processes the human's response and continues

## What You'll Build

| Feature | MAF API | Purpose |
|---------|---------|--------|
| Explicit HITL gate | `ctx.request_info(data, type)` | Pause workflow, show data to human |
| Response handler | `@response_handler` | Process the human's answer |
| Resume | `workflow.run(responses={request_id: answer})` | Continue after human decision |
| Tool approval | `@tool(approval_mode="always_require")` | Per-tool-call approval |
| Functional HITL | `@workflow` + `ctx.request_info()` | Simpler pattern for linear flows |

## Setup

In [ ]:
import os
import sys
import json
from typing import Any

sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import (
    Agent, AgentExecutor, AgentExecutorRequest, AgentExecutorResponse,
    Executor, Message, WorkflowBuilder, WorkflowContext, WorkflowRunState,
    executor, handler, response_handler, tool,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import restart_pod, scale_service, flush_cache, toggle_feature_flag

load_dotenv("../.env")
print("\u2705 Imports ready")

---
## Step 1: The HITL Pattern

1. Executor calls `await ctx.request_info(data, type)` → workflow **PAUSES**
2. Caller checks `result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS`
3. Caller gets `result.get_request_info_events()` to see what was asked
4. Caller resumes with `workflow.run(responses={request_id: answer})`
5. `@response_handler` method receives the answer

**Critical:** `@response_handler` signature MUST be `(self, original_request, response, ctx)` — 4 params

---
## Step 2: Approval Gate Executor (REFERENCE)

In [ ]:
class ApprovalGate(Executor):
    """Pauses the workflow to ask a human for approval."""
    def __init__(self):
        super().__init__(id="approval-gate")
    
    @handler
    async def handle(self, message: str, ctx: WorkflowContext) -> None:
        ctx.set_state("pending_action", message)
        # PAUSES the workflow here:
        approval = await ctx.request_info(
            f"\U0001f6a8 APPROVAL REQUIRED:\n{message}\n\nApprove? (yes/no)",
            str,
        )
        await ctx.send_message(approval)
    
    @response_handler
    async def on_response(self, original_request: str, response: str, ctx: WorkflowContext) -> None:
        """4 params required: self, original_request, response, ctx"""
        await ctx.send_message(response)

print("\u2705 ApprovalGate defined")

In [ ]:
class ExecuteAction(Executor):
    """Executes or aborts based on approval."""
    def __init__(self):
        super().__init__(id="execute-action")
    
    @handler
    async def handle(self, message: str, ctx: WorkflowContext) -> None:
        action = ctx.get_state("pending_action")
        if "yes" in message.lower():
            await ctx.yield_output(f"\u2705 APPROVED — Executing: {action}")
        else:
            await ctx.yield_output(f"\u274c REJECTED — Aborted: {action}")

print("\u2705 ExecuteAction defined")

---
## Step 3: Wire and Run the HITL Workflow (YOUR TURN)

```
ApprovalGate → ExecuteAction
```

In [ ]:
# TODO: Build and run the HITL workflow
#
# 1. gate = ApprovalGate()
#    execute = ExecuteAction()
#
# 2. workflow = (WorkflowBuilder(start_executor=gate)
#        .add_edge(gate, execute)
#        .build())
#
# 3. result = await workflow.run("restart payment-api pod-3")
#    # Check: result.get_final_state() == IDLE_WITH_PENDING_REQUESTS
#
# 4. events = result.get_request_info_events()
#    req_id = events[0].request_id
#    print(f"Human sees: {events[0].data}")
#
# 5. result2 = await workflow.run(responses={req_id: "yes"})
#    print(result2.get_outputs())

gate = ApprovalGate()
execute = ExecuteAction()

# YOUR CODE HERE

In [ ]:
# Validate
assert result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
assert result2.get_final_state() == WorkflowRunState.IDLE
assert "APPROVED" in result2.get_outputs()[0]
print("\u2705 HITL workflow works!")
print(f"   Output: {result2.get_outputs()[0]}")

---
## Step 4: Tool Approval with Agent (YOUR TURN)

Build a workflow where an agent calls approval-gated tools.
Use the approval loop pattern:

```python
while events.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
    for evt in events.get_request_info_events():
        responses[evt.request_id] = evt.data.to_function_approval_response(approved=True)
    events = await workflow.run(responses=responses)
```

In [ ]:
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

remediation_agent = Agent(
    client,
    id="remediation-executor",
    name="RemediationExecutor",
    instructions="Execute remediation. Call restart_pod, scale_service, flush_cache, or toggle_feature_flag as needed.",
    tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag],
)

# TODO: Build workflow with AgentExecutor + approval loop
# @executor(id="prepare")
# async def prepare(plan: str, ctx: WorkflowContext) -> None:
#     msg = Message("user", contents=[plan])
#     await ctx.send_message(AgentExecutorRequest(messages=[msg], should_respond=True))
#
# @executor(id="done")
# async def done(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
#     await ctx.yield_output(f"Done: {response.agent_response.text[:200]}")
#
# agent_exec = AgentExecutor(remediation_agent)
# tool_workflow = (WorkflowBuilder(start_executor=prepare)
#     .add_edge(prepare, agent_exec)
#     .add_edge(agent_exec, done)
#     .build())
#
# # Run with approval loop
# events = await tool_workflow.run("restart_pod for payment-api pod-3, reason: OOM")
# while events.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
#     responses = {}
#     for evt in events.get_request_info_events():
#         responses[evt.request_id] = evt.data.to_function_approval_response(approved=True)
#     events = await tool_workflow.run(responses=responses)
# print(events.get_outputs())

---
## Step 5: Functional Workflow with @workflow (YOUR TURN)

Simpler pattern for linear flows with HITL.

In [ ]:
from agent_framework import RunContext, workflow, step

# TODO: Build a functional workflow
# @workflow
# async def remediate_with_approval(plan: str, ctx: RunContext) -> str:
#     approval = await ctx.request_info(
#         f"Plan: {plan}\nApprove?",
#         response_type=str,
#         request_id="plan_approval",
#     )
#     if "yes" in approval.lower():
#         return f"Executing: {plan}"
#     return f"Aborted: {plan}"
#
# r1 = await remediate_with_approval.run("restart pod-3")
# assert r1.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
# r2 = await remediate_with_approval.run(responses={"plan_approval": "yes"})
# print(r2.get_outputs())

---
## Summary

| Pattern | API | When to Use |
|---------|-----|-------------|
| Explicit pause | `ctx.request_info()` + `@response_handler` | Business decisions, plan review |
| Tool-level approval | `@tool(approval_mode="always_require")` | Dangerous tool calls |
| Workflow resume | `workflow.run(responses={...})` | All HITL patterns |
| Functional HITL | `@workflow` + `ctx.request_info()` | Simple linear flows |

---
## ➡️ Bonus: Challenge 4

[Open Challenge 4 →](../challenge-4/README.md)

# Challenge 3: Human-in-the-Loop & Resilience

## Why HITL Matters

Your workflow routes correctly — but would you trust it to `restart_pod` in production
at 3 AM without asking anyone? What if the diagnostics are wrong?

Production agent systems need:
- **Human approval gates** before destructive actions
- **Tool-level approval** for dangerous operations
- **Workflow pause/resume** so the system can wait for humans
- **Retry logic** when verification fails

MAF provides these via:
- `ctx.request_info()` — pauses the workflow and asks the caller for input
- `@tool(approval_mode="always_require")` — tool calls trigger approval events
- `workflow.run(responses={...})` — resumes a paused workflow with answers

```
Workflow runs ─→ Agent calls restart_pod() ─→ ⏸ PAUSED
                                                  │
                                         Human reviews request
                                                  │
                                         Approves/Rejects
                                                  │
                              workflow.run(responses={...}) ─→ ▶ RESUMES
```

---

## What You'll Build

| Feature | MAF API | Purpose |
|---------|---------|--------|
| Tool approval gate | `@tool(approval_mode="always_require")` | Pause on dangerous tool calls |
| Explicit HITL | `ctx.request_info()` | Ask human a question mid-workflow |
| Resume | `workflow.run(responses={...})` | Continue after human decision |
| Retry loop | Native Python in functional workflow | Retry verification up to N times |

## Setup

In [ ]:
import os
import sys
import json
from typing import Any, Literal

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Case,
    Default,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,
    RunContext,
    executor,
    workflow,
    step,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential

from tools.mock_infra import (
    restart_pod, scale_service, flush_cache, toggle_feature_flag,
    get_health_status, run_smoke_test,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("✅ Imports ready — WorkflowRunState, RunContext, workflow, step loaded")

---
## Step 1: Understand Tool Approval

Look at how tools are defined in `tools/mock_infra.py`:

```python
@tool(approval_mode="always_require")
def restart_pod(service, pod_id, reason) -> dict:
    ...
```

When an agent calls a tool with `approval_mode="always_require"`, MAF:
1. **Pauses** the workflow
2. Emits a `request_info` event containing the function name + arguments
3. **Waits** for the caller to respond with approve/reject
4. Only executes the tool if approved

The remediation tools (`restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`)
ALL require approval — they're destructive operations.

---
## Step 2: Models

In [ ]:
class RemediationPlan(BaseModel):
    action: Literal["restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate"]
    target_service: str
    target_details: str
    risk_level: Literal["low", "medium", "high"]
    estimated_downtime_seconds: int
    rollback_strategy: str
    requires_approval: bool

print("✅ RemediationPlan model defined")

---
## Step 3: Build the Remediation Executor Agent

This agent **executes** the remediation plan using tools that require approval.
When it calls `restart_pod(...)`, MAF will pause the workflow.

In [ ]:
client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["FOUNDRY_MODEL"],
    credential=AzureCliCredential(),
)

remediation_executor_agent = Agent(
    client,
    id="remediation-executor",
    name="RemediationExecutor",
    instructions=(
        "You are a Remediation Executor. You receive a remediation plan and execute it.\n"
        "\n"
        "IMPORTANT: Your tools require human approval before they execute.\n"
        "Call the appropriate tool based on the plan's action field:\n"
        "- restart_pod: for pod restarts\n"
        "- scale_service: for scaling replicas\n"
        "- flush_cache: for cache issues\n"
        "- toggle_feature_flag: for feature flag changes\n"
        "\n"
        "Always provide a clear 'reason' parameter for the audit trail.\n"
        "After tool execution, confirm what was done."
    ),
    tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag],
)

print("✅ Remediation executor agent created")
print("   Tools: restart_pod, scale_service, flush_cache, toggle_feature_flag")
print("   All tools have approval_mode='always_require'")

---
## Step 4: Build the HITL Workflow (YOUR TURN)

Build a workflow that:
1. Receives a remediation plan (JSON string)
2. Sends it to the remediation executor agent
3. The agent calls a tool → workflow **pauses** for approval
4. You approve/reject → workflow **resumes**

### Executors

In [ ]:
@executor(id="prepare_remediation")
async def prepare_remediation(plan_json: str, ctx: WorkflowContext) -> None:
    """Parse remediation plan and forward to executor agent."""
    plan = RemediationPlan.model_validate_json(plan_json)
    ctx.set_state("plan", plan)
    
    user_msg = Message("user", contents=[
        f"Execute this remediation plan:\n"
        f"Action: {plan.action}\n"
        f"Target: {plan.target_service}\n"
        f"Details: {plan.target_details}\n"
        f"Risk: {plan.risk_level}\n"
        f"Reason: Fix for production incident"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


@executor(id="confirm_execution")
async def confirm_execution(response: AgentExecutorResponse, ctx: WorkflowContext) -> None:
    """Terminal: confirm remediation completed."""
    plan: RemediationPlan = ctx.get_state("plan")
    await ctx.yield_output(
        f"✅ Remediation executed:\n"
        f"   Action: {plan.action}\n"
        f"   Target: {plan.target_details}\n"
        f"   Agent response: {response.agent_response.text[:200]}"
    )

print("✅ Executors defined")

### Wire the HITL Workflow

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  TODO: Wire the HITL workflow                                   ║
# ║                                                                 ║
# ║  Graph:                                                         ║
# ║    prepare_remediation → remediation_agent → confirm_execution  ║
# ║                                                                 ║
# ║  Steps:                                                         ║
# ║    1. remediation_exec = AgentExecutor(remediation_executor_agent)║
# ║    2. hitl_workflow = (                                          ║
# ║           WorkflowBuilder(start_executor=prepare_remediation)    ║
# ║           .add_edge(prepare_remediation, remediation_exec)       ║
# ║           .add_edge(remediation_exec, confirm_execution)         ║
# ║           .build()                                               ║
# ║       )                                                          ║
# ╚════════════════════════════════════════════════════════════════╝

# YOUR CODE HERE
remediation_exec = AgentExecutor(remediation_executor_agent)

hitl_workflow = None  # Replace with your workflow

---
## Step 5: Run with Approval Loop (YOUR TURN)

When you run this workflow, it will **pause** when the agent tries to call
`restart_pod()`. You need to handle the approval and resume.

### The Approval Loop Pattern:
```python
# Phase 1: Run until paused
result = await hitl_workflow.run(plan_json)

# Phase 2: Process approval requests in a loop
while result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
    request_info_events = result.get_request_info_events()
    responses = {}
    for event in request_info_events:
        print(f"  Approval requested: {event.data}")
        # Approve it:
        responses[event.request_id] = event.data.to_function_approval_response(approved=True)
    # Resume the workflow
    result = await hitl_workflow.run(responses=responses)

# Phase 3: Get final output
print(result.get_outputs())
```

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  TODO: Run the workflow and handle the approval loop            ║
# ║                                                                 ║
# ║  1. Create the plan JSON (use the sample below)                 ║
# ║  2. Run: result = await hitl_workflow.run(plan_json)            ║
# ║  3. Check: result.get_final_state() should be                   ║
# ║     IDLE_WITH_PENDING_REQUESTS (paused for approval)            ║
# ║  4. Loop: while paused, approve all requests and resume         ║
# ║  5. Print the final output                                      ║
# ╚════════════════════════════════════════════════════════════════╝

plan_json = json.dumps({
    "action": "restart_pod",
    "target_service": "payment-api",
    "target_details": "payment-api-pod-3",
    "risk_level": "medium",
    "estimated_downtime_seconds": 8,
    "rollback_strategy": "Scale to 4 replicas if restart fails",
    "requires_approval": True,
})

# YOUR CODE HERE: Run the workflow and handle the approval loop
# result = await hitl_workflow.run(plan_json)
# while result.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS:
#     ...

In [ ]:
# ✅ Validate HITL workflow completed
final_outputs = result.get_outputs()
assert final_outputs, "Workflow should complete after approval"
print(f"Final output: {final_outputs[0]}")
print("✅ HITL workflow validation passed — approval loop works!")

---
## Step 6: Functional Workflow with `ctx.request_info()` (YOUR TURN)

MAF also supports **functional workflows** — plain Python async functions
with `ctx.request_info()` to pause for human input.

This is a higher-level pattern: instead of tool-level approval,
you explicitly pause the workflow and ask the human a question.

```python
from agent_framework import RunContext, workflow, step

@workflow
async def my_workflow(input: str, ctx: RunContext) -> str:
    result = await some_processing(input)
    
    # PAUSE: ask the human for approval
    approval = await ctx.request_info(
        {"plan": result, "question": "Approve this action?"},
        response_type=str,
        request_id="approval_gate",
    )
    
    if approval == "approved":
        return f"Executing: {result}"
    else:
        return "Aborted by human"
```

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  TODO: Build a functional workflow with explicit HITL            ║
# ║                                                                 ║
# ║  @workflow                                                      ║
# ║  async def remediate_with_approval(plan_json: str,              ║
# ║                                    ctx: RunContext) -> str:     ║
# ║      1. Parse the RemediationPlan from plan_json                ║
# ║      2. Use ctx.request_info() to show the plan to the human   ║
# ║         and ask for approval (response_type=str)                ║
# ║      3. If approval == "yes": return "Executing: {plan.action}" ║
# ║         Else: return "Aborted: human rejected"                  ║
# ║                                                                 ║
# ║  Then run it:                                                   ║
# ║    result1 = await remediate_with_approval.run(plan_json)       ║
# ║    # Check state is IDLE_WITH_PENDING_REQUESTS                  ║
# ║    # Resume: await remediate_with_approval.run(                 ║
# ║    #     responses={"plan_review": "yes"})                      ║
# ╚════════════════════════════════════════════════════════════════╝

@workflow
async def remediate_with_approval(plan_json: str, ctx: RunContext) -> str:
    """Functional workflow with explicit HITL gate."""
    # YOUR CODE HERE
    pass

In [ ]:
# Run the functional workflow
print("\U0001f680 Running functional HITL workflow...")

result1 = await remediate_with_approval.run(plan_json)
print(f"   State after first run: {result1.get_final_state()}")

# Check it paused
assert result1.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
requests = result1.get_request_info_events()
print(f"   Pending request: {requests[0].request_id}")
print(f"   Data: {requests[0].data}")

# Resume with approval
print("\n   ✅ Human approves...")
result2 = await remediate_with_approval.run(responses={requests[0].request_id: "yes"})
print(f"   Final state: {result2.get_final_state()}")
print(f"   Output: {result2.get_outputs()[0]}")

In [ ]:
assert result2.get_final_state() == WorkflowRunState.IDLE
assert result2.get_outputs(), "Should produce output after approval"
print("✅ Functional HITL workflow validation passed")

---
## Step 7: Retry Pattern (YOUR TURN)

After remediation, verification might fail. A production system retries before escalating.
In functional workflows, you just use native Python loops — no special graph constructs.

In [ ]:
# ╔════════════════════════════════════════════════════════════════╗
# ║  TODO: Build a retry workflow                                   ║
# ║                                                                 ║
# ║  @workflow                                                      ║
# ║  async def remediate_with_retry(service: str,                   ║
# ║                                 ctx: RunContext) -> str:        ║
# ║      max_retries = 3                                            ║
# ║      for attempt in range(max_retries):                         ║
# ║          health = get_health_status(service=service)            ║
# ║          if "ready: true" in health.lower():                    ║
# ║              return f"Fixed on attempt {attempt + 1}"           ║
# ║      return f"ESCALATION: {service} still unhealthy"            ║
# ║                                                                 ║
# ║  This shows how native Python control flow works in             ║
# ║  functional workflows — no special graph constructs needed.    ║
# ╚════════════════════════════════════════════════════════════════╝

@workflow
async def remediate_with_retry(service: str, ctx: RunContext) -> str:
    """Retry verification up to 3 times, then escalate."""
    # YOUR CODE HERE
    pass

In [ ]:
# Test the retry workflow
result = await remediate_with_retry.run("payment-api")
outputs = result.get_outputs()
print(f"Result: {outputs[0]}")
assert outputs, "Should produce output"
print("✅ Retry workflow works")

---
## Summary: HITL Patterns

| Pattern | API | When to Use |
|---------|-----|-------------|
| Tool-level approval | `@tool(approval_mode="always_require")` | Dangerous tool calls |
| Explicit HITL gate | `ctx.request_info()` | Business decisions, plan review |
| Workflow pause/resume | `workflow.run(responses={...})` | Any async human interaction |
| Retry with escalation | Loops in functional workflows | Verification, health checks |

These patterns make the difference between a **demo** and a **production system**.

---

## ➡️ Bonus: Challenge 4 — Advanced Composition

If you have time remaining, head to Challenge 4 for:
- Workflow-as-Agent pattern (expose entire pipeline as a callable agent)
- Sub-workflow composition
- OpenTelemetry observability
- Parallel fan-out for multi-service incidents

[Open Challenge 4 →](../challenge-4/README.md)

# Challenge 3: Human-in-the-Loop & Resilience

## Why HITL Matters

Your workflow routes correctly — but would you trust it to `restart_pod` in production
at 3 AM without asking anyone? What if the diagnostics are wrong?

Production agent systems need:
- **Human approval gates** before destructive actions
- **Tool-level approval** for dangerous operations
- **Retry logic** when verification fails
- **Workflow pause/resume** so the system can wait for humans

MAF provides these via:
- `ctx.request_info()` — pauses the workflow and asks the caller for input
- `@tool(approval_mode="always_require")` — tool calls trigger approval events
- `workflow.run(responses={...})` — resumes a paused workflow with answers

```
Workflow runs ─→ Agent calls restart_pod() ─→ ⏸ PAUSED
                                                  │
                                         Human reviews plan
                                                  │
                                         Approves/Rejects
                                                  │
                              workflow.run(responses={...}) ─→ ▶ RESUMES
```

---

## What You'll Build

| Feature | MAF API | Purpose |
|---------|---------|--------|
| Approval gate | `ctx.request_info()` | Pause workflow before remediation |
| Tool approval | `approval_mode="always_require"` | Per-tool-call approval |
| Resume | `workflow.run(responses={...})` | Continue after human decision |
| Retry loop | Edge condition + counter | Retry verification up to 3x |

## Setup

In [ ]:
import os
import sys
import json
from typing import Annotated, Any, Literal
from dataclasses import dataclass

sys.path.insert(0, "..")
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agent_framework import (
    Agent,
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Content,
    Case,
    Default,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from agent_framework.openai import OpenAIChatOptions
from azure.identity import AzureCliCredential
from typing_extensions import Never

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket,
)

load_dotenv("../.env")

with open("../data/incidents.json") as f:
    incidents = json.load(f)

print("\u2705 Imports ready — including Content, WorkflowRunState for HITL")

---
## Step 1: Understand Tool Approval

Look at how tools are defined in `tools/mock_infra.py`:

```python
@tool(approval_mode="always_require")  # ← THIS
def restart_pod(service, pod_id, reason) -> dict:
    ...
```

When an agent calls a tool with `approval_mode="always_require"`, MAF:
1. **Pauses** the workflow
2. Emits a `request_info` event containing the function name + arguments
3. **Waits** for the caller to respond with approve/reject
4. Only executes the tool if approved

The remediation tools (`restart_pod`, `scale_service`, `flush_cache`, `toggle_feature_flag`)
ALL require approval — they're destructive operations.

---
## Step 2: Models (from previous challenges)

In [ ]:
class TriageResult(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    is_recurring: bool
    auto_remediation_allowed: bool
    root_cause_hypothesis: str
    recommended_action: str
    escalation_threshold_minutes: int

class DiagnosticsResult(BaseModel):
    root_cause: str
    evidence: list[str]
    affected_components: list[str]
    confidence: float
    recommended_fix: str
    requires_restart: bool

class RemediationPlan(BaseModel):
    action: Literal["restart_pod", "scale_service", "flush_cache", "toggle_feature_flag", "escalate"]
    target_service: str
    target_details: str
    risk_level: Literal["low", "medium", "high"]
    estimated_downtime_seconds: int
    rollback_strategy: str
    requires_approval: bool

print("\u2705 Models defined")

---
## Step 3: Build the Remediation Executor Agent

This agent **executes** the remediation plan using tools that require approval.
When it calls `restart_pod(...)`, MAF will pause the workflow.

In [ ]:
def make_client() -> FoundryChatClient:
    return FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["FOUNDRY_MODEL"],
        credential=AzureCliCredential(),
    )

def create_remediation_executor_agent() -> Agent:
    """Agent that EXECUTES remediation using approval-gated tools."""
    return Agent(
        client=make_client(),
        name="RemediationExecutor",
        instructions=(
            "You are a Remediation Executor. You receive a remediation plan and execute it.\n"
            "\n"
            "IMPORTANT: Your tools require human approval before they execute.\n"
            "Call the appropriate tool based on the plan's action field:\n"
            "- restart_pod: for pod restarts\n"
            "- scale_service: for scaling replicas\n"
            "- flush_cache: for cache issues\n"
            "- toggle_feature_flag: for feature flag changes\n"
            "\n"
            "Always provide a clear 'reason' parameter for the audit trail.\n"
            "After tool execution, confirm what was done."
        ),
        tools=[restart_pod, scale_service, flush_cache, toggle_feature_flag],
    )

print("\u2705 Remediation executor agent factory defined")
print("   Tools: restart_pod, scale_service, flush_cache, toggle_feature_flag")
print("   All tools have approval_mode='always_require'")

---
## Step 4: Build the HITL Workflow (YOUR TURN)

Build a workflow that:
1. Receives a remediation plan
2. Sends it to the remediation executor agent
3. The agent calls a tool → workflow **pauses** for approval
4. You approve/reject → workflow **resumes**

### Executor to prepare the request

In [ ]:
@executor(id="prepare_remediation")
async def prepare_remediation(plan_json: str, ctx: WorkflowContext[AgentExecutorRequest]) -> None:
    """Parse remediation plan and forward to executor agent."""
    plan = RemediationPlan.model_validate_json(plan_json)
    ctx.set_state("plan", plan)
    
    user_msg = Message("user", contents=[
        f"Execute this remediation plan:\n"
        f"Action: {plan.action}\n"
        f"Target: {plan.target_service}\n"
        f"Details: {plan.target_details}\n"
        f"Risk: {plan.risk_level}\n"
        f"Reason: Fix for production incident"
    ])
    await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


@executor(id="confirm_execution")
async def confirm_execution(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Terminal: confirm remediation completed."""
    plan: RemediationPlan = ctx.get_state("plan")
    await ctx.yield_output(
        f"\u2705 Remediation executed:\n"
        f"   Action: {plan.action}\n"
        f"   Target: {plan.target_details}\n"
        f"   Agent response: {response.agent_response.text[:200]}"
    )

print("\u2705 Executors defined")

### Wire the HITL Workflow

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  TODO: Wire the HITL workflow                                   ║
# ║                                                                 ║
# ║  Graph:                                                         ║
# ║    prepare_remediation → remediation_agent → confirm_execution  ║
# ║                                                                 ║
# ║  Steps:                                                         ║
# ║    1. remediation_agent = AgentExecutor(                        ║
# ║           create_remediation_executor_agent())                  ║
# ║    2. Build workflow with WorkflowBuilder                       ║
# ║    3. Three edges:                                              ║
# ║       prepare_remediation → remediation_agent                   ║
# ║       remediation_agent → confirm_execution                     ║
# ╚══════════════════════════════════════════════════════════════════╝

remediation_agent_executor = AgentExecutor(create_remediation_executor_agent())

hitl_workflow = (
    # YOUR CODE HERE
    # WorkflowBuilder(start_executor=prepare_remediation)
    # .add_edge(...)
    # .add_edge(...)
    # .build()
    None  # Replace with your workflow
)

---
## Step 5: Run with Approval Loop (YOUR TURN)

When you run this workflow, it will **pause** when the agent tries to call
`restart_pod()`. You need to handle the approval request and resume.

### The Approval Loop Pattern:
```python
# Phase 1: Run until paused
events = await workflow.run(input_data)
request_info_events = events.get_request_info_events()

# Phase 2: Process each approval request
while request_info_events:
    responses = {}
    for event in request_info_events:
        data = event.data  # Contains the tool call details
        # Approve it:
        responses[event.request_id] = data.to_function_approval_response(approved=True)
    
    # Resume the workflow with responses
    events = await workflow.run(responses=responses)
    request_info_events = events.get_request_info_events()
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  TODO: Run the HITL workflow with the approval loop             ║
# ║                                                                 ║
# ║  1. Create a sample remediation plan (JSON string):             ║
# ║     {"action": "restart_pod",                                   ║
# ║      "target_service": "payment-api",                           ║
# ║      "target_details": "payment-api-pod-3",                     ║
# ║      "risk_level": "medium",                                    ║
# ║      "estimated_downtime_seconds": 8,                           ║
# ║      "rollback_strategy": "Scale to 4 replicas if restart fails",║
# ║      "requires_approval": true}                                 ║
# ║                                                                 ║
# ║  2. Run: events = await hitl_workflow.run(plan_json)            ║
# ║                                                                 ║
# ║  3. Check state: events.get_final_state() should be            ║
# ║     IDLE_WITH_PENDING_REQUESTS (paused for approval)            ║
# ║                                                                 ║
# ║  4. Process approvals in a while loop (see pattern above)       ║
# ║                                                                 ║
# ║  5. Print the final output                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

plan_json = json.dumps({
    "action": "restart_pod",
    "target_service": "payment-api",
    "target_details": "payment-api-pod-3",
    "risk_level": "medium",
    "estimated_downtime_seconds": 8,
    "rollback_strategy": "Scale to 4 replicas if restart fails",
    "requires_approval": True,
})

# YOUR CODE HERE: Run the workflow and handle the approval loop
# ...
# Print approvals as they come in:
# print(f"  Approval requested for: {data.function_call.name}")
# print(f"  Arguments: {data.function_call.parse_arguments()}")
# print(f"  → AUTO-APPROVING for demo")

In [ ]:
# Validate HITL workflow completed
final_outputs = events.get_outputs()
assert final_outputs, "Workflow should complete after approval"
assert "restart_pod" in final_outputs[0] or "Remediation" in final_outputs[0]
print("\u2705 HITL workflow validation passed — approval loop works!")

---
## Step 6: Reject an Approval (Observe Behavior)

What happens when you **reject** a tool call? The agent receives an error
and must adapt — perhaps choosing a less risky action or escalating.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  TODO (Exploration): Run the same workflow but REJECT the       ║
# ║  first approval request                                         ║
# ║                                                                 ║
# ║  Change: data.to_function_approval_response(approved=False)     ║
# ║                                                                 ║
# ║  Observe: Does the agent try a different tool?                  ║
# ║  Does it escalate? What's in the final output?                  ║
# ╚══════════════════════════════════════════════════════════════════╝

# Optional exploration:
# events_reject = await hitl_workflow.run(plan_json)
# ...

---
## Step 7: Functional Workflow with `ctx.request_info()` (YOUR TURN)

MAF also supports **functional workflows** — plain Python async functions
with `ctx.request_info()` to pause for human input.

This is a higher-level pattern: instead of tool-level approval,
you explicitly pause the workflow and ask the human a question.

```python
from agent_framework import RunContext, workflow, step

@workflow
async def my_workflow(input: str, ctx: RunContext) -> str:
    result = await some_agent_call(input)
    
    # PAUSE: ask the human for approval
    approval = await ctx.request_info(
        {"plan": result, "question": "Approve this action?"},
        response_type=str,
        request_id="approval_gate",
    )
    
    if approval == "approved":
        return await execute(result)
    else:
        return "Aborted by human"
```

In [ ]:
from agent_framework import RunContext, workflow, step

# ╔══════════════════════════════════════════════════════════════════╗
# ║  TODO: Build a functional workflow with explicit HITL            ║
# ║                                                                 ║
# ║  @workflow                                                      ║
# ║  async def remediate_with_approval(plan_json: str,              ║
# ║                                    ctx: RunContext) -> str:     ║
# ║      1. Parse the RemediationPlan from plan_json                ║
# ║      2. Use ctx.request_info() to show the plan to the human   ║
# ║         and ask for approval (response_type=bool)               ║
# ║      3. If approved: return "Executing: {plan.action}"          ║
# ║         If rejected: return "Aborted: human rejected"           ║
# ║                                                                 ║
# ║  Then run it:                                                   ║
# ║    result1 = await remediate_with_approval.run(plan_json)       ║
# ║    # Check state is IDLE_WITH_PENDING_REQUESTS                  ║
# ║    # Resume with: await remediate_with_approval.run(            ║
# ║    #     responses={"approval_gate": True})                     ║
# ╚══════════════════════════════════════════════════════════════════╝

@workflow
async def remediate_with_approval(plan_json: str, ctx: RunContext) -> str:
    """Functional workflow with explicit HITL gate."""
    # YOUR CODE HERE
    pass

In [ ]:
# Run the functional workflow
print("\U0001f680 Running functional HITL workflow...")

result1 = await remediate_with_approval.run(plan_json)
print(f"   State after first run: {result1.get_final_state()}")

# Check it paused
assert result1.get_final_state() == WorkflowRunState.IDLE_WITH_PENDING_REQUESTS
requests = result1.get_request_info_events()
print(f"   Pending request: {requests[0].request_id}")
print(f"   Data: {requests[0].data}")

# Resume with approval
print("\n   \u2705 Human approves...")
result2 = await remediate_with_approval.run(responses={"approval_gate": True})
print(f"   Final state: {result2.get_final_state()}")
print(f"   Output: {result2.get_outputs()[0]}")

In [ ]:
assert result2.get_final_state() == WorkflowRunState.IDLE
assert result2.get_outputs(), "Should produce output after approval"
print("\u2705 Functional HITL workflow validation passed")

---
## Step 8: Retry Pattern with `@step` (YOUR TURN)

After remediation, verification might fail. A production system retries
before escalating. MAF's `@step` decorator saves results so they don't
re-execute on resume.

Build a workflow that:
1. Executes remediation
2. Runs verification
3. If verification fails: retry up to 3 times
4. If still failing: escalate to human

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  TODO: Build a retry workflow                                   ║
# ║                                                                 ║
# ║  @workflow                                                      ║
# ║  async def remediate_with_retry(service: str,                   ║
# ║                                 ctx: RunContext) -> str:        ║
# ║      max_retries = 3                                            ║
# ║      for attempt in range(max_retries):                         ║
# ║          # Simulate remediation (just call get_health_status)   ║
# ║          health = get_health_status(service=service)            ║
# ║          if health["ready"]:                                    ║
# ║              return f"Fixed on attempt {attempt + 1}"           ║
# ║          # else: loop again                                     ║
# ║      # All retries exhausted                                    ║
# ║      return f"ESCALATION: {service} still unhealthy"            ║
# ║                                                                 ║
# ║  This demonstrates how native Python control flow works in      ║
# ║  functional workflows — no special graph constructs needed.     ║
# ╚══════════════════════════════════════════════════════════════════╝

@workflow
async def remediate_with_retry(service: str, ctx: RunContext) -> str:
    """Retry verification up to 3 times, then escalate."""
    # YOUR CODE HERE
    pass

In [ ]:
# Test the retry workflow
result = await remediate_with_retry.run("payment-api")
outputs = result.get_outputs()
print(f"Result: {outputs[0]}")
assert outputs, "Should produce output"
print("\u2705 Retry workflow works")

---
## Summary: What You've Learned

| Pattern | API | When to Use |
|---------|-----|-------------|
| Tool-level approval | `@tool(approval_mode="always_require")` | Dangerous tool calls |
| Explicit HITL gate | `ctx.request_info()` | Business decisions, plan review |
| Workflow pause/resume | `workflow.run(responses={...})` | Any async human interaction |
| Retry with escalation | Loops in functional workflows | Verification, health checks |
| Result caching | `@step` decorator | Expensive ops that shouldn't re-run |

These patterns make the difference between a **demo** and a **production system**.

---
## \u27a1\ufe0f Bonus: Challenge 4 — Advanced Composition

If you have time remaining, head to Challenge 4 for:
- Workflow-as-Agent pattern (expose entire pipeline as a callable agent)
- Sub-workflow composition (remediation as a nested workflow)
- OpenTelemetry observability
- Parallel fan-out for multi-service incidents

[Open Challenge 4 →](../challenge-4/README.md)

# 🧠 Step 4: Memory Patterns

## Making the System Learn

Our workflow handles incidents — but every time it starts from scratch.
A human SRE gets better over time because they **remember** past incidents.

In this step, we'll add memory so our agents can:
- Store resolutions from past incidents
- Retrieve relevant context when similar issues occur
- Build institutional knowledge that persists

## Key Concept: Context Providers

MAF provides `ContextProvider` — a way to inject relevant context into an agent's
prompt at runtime. The agent doesn't need to "remember" everything; it gets the
relevant memories injected automatically.

```
Alert fires → Memory Provider retrieves similar past incidents
            → Injects resolution context into Triage Agent prompt
            → Agent says: "This is the same as INC-12345, fixed by restarting pod-3"
```

In [ ]:
import os
import sys
import json
from datetime import datetime
from dataclasses import dataclass, field
sys.path.insert(0, "..")
from dotenv import load_dotenv

from agent_framework import ChatAgent
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential

from tools.mock_infra import (
    check_alert_history, get_runbook,
    get_metrics, get_logs, check_dependencies,
    restart_pod, scale_service, flush_cache, toggle_feature_flag, escalate_to_human,
    get_health_status, run_smoke_test,
    post_to_slack, create_incident_ticket, update_status_page,
)

load_dotenv("../.env")
print("✅ Imports ready")

## Build an Incident Memory Store

We'll create a simple in-memory store that holds past incident resolutions.
In production, this would be backed by Azure AI Search, CosmosDB, or Redis.

### 🎯 YOUR TASK
Implement the `search` method to find relevant past incidents based on the
service name and keywords.

In [ ]:
@dataclass
class IncidentMemory:
    """Simple incident resolution memory store."""
    service: str
    title: str
    root_cause: str
    resolution: str
    timestamp: str
    time_to_resolve_minutes: int


class IncidentMemoryStore:
    """In-memory store for past incident resolutions.
    
    In production, you'd back this with Azure AI Search for semantic retrieval
    or CosmosDB for structured queries.
    """
    
    def __init__(self):
        # Pre-seed with some "past" incidents
        self.memories: list[IncidentMemory] = [
            IncidentMemory(
                service="payment-api",
                title="High latency on payment-api due to memory leak",
                root_cause="PaymentBatchProcessor leaks memory during bulk processing. Pod-3 hit OOM at 4096Mi.",
                resolution="Restarted pod-3. Memory dropped from 3.9GB to 512MB. Latency recovered in 2 minutes. JIRA-4521 tracks permanent fix.",
                timestamp="2026-06-27T03:15:00Z",
                time_to_resolve_minutes=8,
            ),
            IncidentMemory(
                service="payment-api",
                title="Payment API connection pool exhaustion",
                root_cause="Burst of retry storms from order-service exhausted the 50-connection HikariCP pool.",
                resolution="Restarted pod-3 to reset connections. Increased pool size to 75 in next deploy.",
                timestamp="2026-06-25T14:22:00Z",
                time_to_resolve_minutes=12,
            ),
            IncidentMemory(
                service="notification-service",
                title="Email delivery failures - provider rate limiting",
                root_cause="Primary email provider rate-limited us at 1000 emails/hour. Marketing campaign triggered 5000 emails.",
                resolution="Toggled feature flag 'use_backup_email' to route through backup provider. Queue cleared in 15 minutes.",
                timestamp="2026-06-28T19:45:00Z",
                time_to_resolve_minutes=5,
            ),
            IncidentMemory(
                service="user-service",
                title="TLS handshake failures after cert expiry",
                root_cause="Let's Encrypt certificate for user-service.internal expired. cert-manager renewal failed due to DNS challenge timeout.",
                resolution="Manual cert renewal by security team. Auto-remediation NOT possible for cert issues.",
                timestamp="2026-06-05T08:30:00Z",
                time_to_resolve_minutes=35,
            ),
        ]
    
    def search(self, service: str, keywords: str = "") -> list[IncidentMemory]:
        """Find relevant past incidents for a service.
        
        In production, this would use semantic search (embeddings).
        For the workshop, we do simple keyword matching.
        """
        results = []
        keywords_lower = keywords.lower()
        
        for memory in self.memories:
            # Match by service
            if memory.service == service:
                results.append(memory)
            # Also match by keyword in root cause or title
            elif keywords_lower and (
                keywords_lower in memory.root_cause.lower()
                or keywords_lower in memory.title.lower()
            ):
                results.append(memory)
        
        return results
    
    def store(self, memory: IncidentMemory):
        """Store a new incident resolution in memory."""
        self.memories.append(memory)
        print(f"💾 Stored new memory: {memory.title}")


# Initialize the memory store
memory_store = IncidentMemoryStore()
print(f"✅ Memory store initialized with {len(memory_store.memories)} past incidents")

## Memory-Enhanced Triage Agent

Now let's create a Triage Agent that checks memory FIRST — before doing anything else.
If it finds a similar past incident, it can fast-track the response.

### 🎯 YOUR TASK
Create a tool function `search_incident_memory` that wraps the memory store,
then use it in the triage agent.

In [ ]:
from typing import Annotated
from pydantic import Field

# Create a tool function that wraps the memory store
def search_incident_memory(
    service_name: Annotated[str, Field(description="The service to search past incidents for")],
    keywords: Annotated[str, Field(description="Keywords to search for (e.g., 'memory leak', 'rate limit')")] = "",
) -> str:
    """Search the incident memory store for similar past incidents.
    Returns past incidents with their root causes and resolutions."""
    
    results = memory_store.search(service_name, keywords)
    
    if not results:
        return json.dumps({"found": 0, "message": "No similar past incidents found."})
    
    past_incidents = []
    for r in results:
        past_incidents.append({
            "title": r.title,
            "root_cause": r.root_cause,
            "resolution": r.resolution,
            "when": r.timestamp,
            "time_to_resolve_minutes": r.time_to_resolve_minutes,
        })
    
    return json.dumps({"found": len(past_incidents), "past_incidents": past_incidents}, indent=2)


print("✅ search_incident_memory tool created")
# Quick test
print("\nTest search for 'payment-api':")
print(search_incident_memory("payment-api", "memory"))

In [ ]:
async def memory_enhanced_triage():
    """Triage agent that leverages past incident memory."""
    
    with open("data/incidents.json") as f:
        incidents = json.load(f)
    alert = incidents[0]  # payment-api
    
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as client,
    ):
        # Memory-enhanced triage agent has an EXTRA tool: search_incident_memory
        agent = client.create_agent(
            name="MemoryTriageAgent",
            instructions="""You are a Memory-Enhanced Triage Agent. You have access to a database of past incidents.

When triaging a new alert:
1. FIRST: Use search_incident_memory to check if a similar incident occurred before
2. THEN: Use check_alert_history for recent alert patterns
3. THEN: Use get_runbook for the standard procedure

If you find a matching past incident, reference it explicitly:
- "This matches past incident from [date]: [title]"
- "Previous resolution was: [what worked before]"
- "Expected TTR based on history: [X minutes]"

This helps the Diagnostics and Remediation agents skip unnecessary investigation.""",
            tools=[search_incident_memory, check_alert_history, get_runbook],
        )
        
        result = await agent.run(
            f"New alert:\nTitle: {alert['title']}\nService: {alert['service']}\n"
            f"Severity: {alert['severity']}\nDescription: {alert['description']}\n\n"
            f"Triage this, checking memory first."
        )
        
        print("\n📋 MEMORY-ENHANCED TRIAGE:\n")
        print(result.text)
        return result.text

triage_result = await memory_enhanced_triage()

## Storing New Resolutions

After an incident is resolved, we store the resolution in memory so future incidents benefit.
This creates a **learning loop**.

In [ ]:
# Simulate storing a new resolution after the workflow completes
new_memory = IncidentMemory(
    service="payment-api",
    title="High latency on payment-api - OOM on pod-3 (recurring)",
    root_cause="PaymentBatchProcessor memory leak (3rd occurrence). Pod-3 OOMKilled at 4096Mi limit. Connection pool cascading to order-service.",
    resolution="Restarted payment-api-pod-3. Memory reset to 512MB. Latency recovered. JIRA-4521 priority increased to P1.",
    timestamp=datetime.now().isoformat(),
    time_to_resolve_minutes=6,
)

memory_store.store(new_memory)
print(f"\n📊 Memory store now has {len(memory_store.memories)} incidents")
print(f"   Next time this happens, the agent will resolve even faster.")

## Compare: With vs Without Memory

Notice the difference:

| Without Memory (Step 2) | With Memory (Step 4) |
|---|---|
| Investigates from scratch every time | Immediately recognizes recurring issue |
| Triage takes longer | Fast-tracks known patterns |
| No historical TTR estimate | "Expected resolution: 6-8 minutes" |
| No link to JIRA tickets | "Related: JIRA-4521" |
| Remediation agent may try wrong fix first | Jumps straight to known-good fix |

---

## 🎓 Production Patterns

In a real system, you'd enhance this with:

1. **Azure AI Search** for semantic memory retrieval (not just keyword matching)
2. **Embeddings** to find similar incidents even with different wording
3. **TTL/decay** to phase out outdated memories (infra changes over time)
4. **Confidence scoring** to weight recent resolutions higher
5. **Human feedback loop** — SREs mark if the suggested resolution was actually helpful

## 🏁 Workshop Complete!

### What You Built Today

```
Step 1: Single Agent (Copilot)     → Saw the limitations
Step 2: Specialized Agents + Tools → Task decomposition & tool integration
Step 3: Workflow Orchestration      → Agent coordination with routing
Step 4: Memory Patterns            → Learning system that improves over time
```

### The Journey: Copilot → Orchestrated Agents

| Copilot (Step 1) | Orchestrated Agents (Steps 2-4) |
|---|---|
| Generic advice | Takes real actions |
| No tools | 15 infrastructure tools |
| No memory | Learns from past incidents |
| No verification | Confirms fix worked |
| Can't escalate | Smart routing to humans |
| Same quality forever | Gets better over time |

### Next Steps (After the Workshop)

- **Human-in-the-loop**: Add approval gates before destructive actions
- **Checkpointing**: Persist workflow state for long-running incidents
- **MCP integration**: Connect to real monitoring tools via MCP protocol
- **Observability**: Add OpenTelemetry tracing to monitor agent decisions
- **Evaluation**: Test with red-team scenarios to verify safety guardrails

### Resources

- [Microsoft Agent Framework Docs](https://learn.microsoft.com/en-us/agent-framework/)
- [Agent Framework GitHub](https://github.com/microsoft/agent-framework)
- [Azure AI Foundry](https://ai.azure.com)
- This workshop repo: [github.com/ishasalania/maf-lab](https://github.com/ishasalania/maf-lab)